# Phase 2: Baseline Player Detection and Tracking on SoccerTrack v2

**Course:** SBES 426, Medical Pattern Recognition  
**Dataset:** SoccerTrack v2 (HF release v1.2, revision `325bd4c57ee8e135ab652f3921d274526ada496b`)  
**Owner:** Member 4 (integration). Section content comes from `docs/phase2/sections/` (one file per member).

**Fast mode** (default) reads saved artefacts from `outputs/`; **full mode** re-runs stages via `scripts/`.  
This notebook contains **narrative and figures only**; all logic lives in `src/soccertrack/`.  
Numbers are always read from `outputs/metrics/*.json` or `outputs/tables/*.csv` — never typed by hand.

In [ ]:
"""Notebook setup — fast mode by default."""
import sys, json, warnings
from pathlib import Path

_cwd = Path('.').resolve()
REPO = _cwd if (_cwd / 'src').exists() else _cwd.parent
if (REPO / 'src').exists() and str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

from soccertrack import config
REPO = config.repo_root()

FAST_MODE = True   # Set to False on the lead machine to re-run stages
OUTPUTS   = REPO / 'outputs'
FIXTURES  = REPO / 'tests' / 'fixtures'

cfg = config.load_config()
config.set_seeds(cfg['project']['seed'])

print(f'Repo root : {REPO}')
print(f'Fast mode : {FAST_MODE}')
print(f'Config SHA: {config.config_sha256()[:12]}…')

## 1. Problem and Dataset

**Goal:** detect and track the 22 players in fixed panoramic football video (SoccerTrack v2).  
Two complete, independent detection+tracking pipelines are compared under one shared evaluation:

| Pipeline | Detector | Tracker |
|---|---|---|
| A (from scratch) | HOG+SVM / Small CNN after segmentation | IoU + Hungarian |
| B (pretrained)   | Tiled COCO YOLOv8n                     | ByteTrack |

**Dataset:** SoccerTrack v2, MOT subset — one 4-minute clip (25 fps, ~4096×1080 px) per match.  
22 players annotated per frame; referees and staff are **not** labelled (→ explainable false positives).  
MOT annotations are **0-based** frame indices; trailing columns are **−1** (no visibility labels).

| Split | Matches | Frames (approx.) |
|---|---|---|
| Train | 118575, 118576, 118577, 118578, 128058 | ~30 000 |
| Val   | 117093 | 5 976 |
| **Test (held-out)** | **128057, 132831** | **~12 000** |

## 2. EDA and Data Quality

In [ ]:
"""Load EDA stats (fast mode: fixture; full mode: real outputs)."""
import json
from soccertrack.visualize import plot_eda

if FAST_MODE or not (OUTPUTS / 'eda' / 'stats.json').exists():
    stats_path = FIXTURES / 'eda_stats_sample.json'
    print('[PLACEHOLDER] Using fixture eda_stats_sample.json — replace with real outputs/eda/stats.json')
else:
    stats_path = OUTPUTS / 'eda' / 'stats.json'

with stats_path.open() as fh:
    stats = json.load(fh)

totals = stats.get('totals', {})
paper  = stats.get('paper_reference', {})

print(f"Our dataset  : {totals.get('matches', '?')} matches, "
      f"{totals.get('frames', '?'):,} frames, {totals.get('boxes', '?'):,} boxes")
print(f"Paper reports: {paper.get('frames', '?'):,} frames, "
      f"{paper.get('boxes', '?'):,} boxes (3 test matches not yet downloaded in Phase 2)")

In [ ]:
"""Generate and display EDA figures."""
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_dir = OUTPUTS / 'figures' / 'eda'
produced = plot_eda(stats, box_sample=None, out_dir=fig_dir)
print(f'Produced {len(produced)} figures in {fig_dir}')

# Display inline
for fig_path in produced:
    img = mpimg.imread(str(fig_path))
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(fig_path.stem.replace('_', ' ').title())
    plt.tight_layout()
    plt.show()

**Key findings from EDA:**
- Frame indices are **0-based** (despite the MOT format documentation stating 1-based).
- All four trailing MOT columns are **−1** (no visibility labels, no confidence).
- Exactly **22 boxes per frame** in every clean clip.
- Median box ≈ **20 × 45 px** in our clips vs. 25 × 55 px in the paper (slightly smaller camera).
- Some clips have slightly fewer frames than 6 000 (truncated recordings).

> **[PLACEHOLDER]** Replace with real per-match numbers from `outputs/eda/stats.json` once Member 1 delivers M2.

## 3. Preprocessing

> **[PLACEHOLDER — Member 1 section]** Insert content from `docs/phase2/sections/member_1_scratch.md` §Preprocessing.

Key policies:
- **No whole-frame downscaling**: players are ~20×45 px; downscaling to e.g. 640 px wide would erase them.
- Approach A: 0.5× scale only for background subtraction (then back to full-frame coords before writing).
- Approach B: native-resolution overlapping tiles.
- Boxes outside the frame are clipped before writing (contract rule).
- All normalisation statistics (colour histograms, background models) are computed on training data only.

## 4. Split and Leakage Controls

In [ ]:
"""Display and verify the match-level split."""
split_cfg = cfg['project']['split']

train  = set(split_cfg.get('train', []))
val    = set(split_cfg.get('val', []))
test   = set(split_cfg.get('test', []))

print('Split table:')
print(f'  train : {sorted(train)}')
print(f'  val   : {sorted(val)}')
print(f'  test  : {sorted(test)}  ← held out')

assert train.isdisjoint(val),  'LEAK: train ∩ val'
assert train.isdisjoint(test), 'LEAK: train ∩ test'
assert val.isdisjoint(test),   'LEAK: val ∩ test'
print('\n✓ Split is leakage-safe (all three sets are disjoint).')

> **[PLACEHOLDER — Member 1 section]** Insert content from `docs/phase2/sections/member_1_scratch.md` §Split.

**Split rationale:** match-level split following the HF README v1.2 protocol.  
Within-match splits would leak: the two halves share teams, camera, weather, and lighting.  
The frozen test protocol requires a single logged run per system per phase (see `outputs/test_log.md`).

## 5. Segmentation and Region Proposals

> **[PLACEHOLDER — Member 1 section]** Insert content from `docs/phase2/sections/member_1_scratch.md` §Segmentation.

Pipeline A segmentation steps:
1. **Pitch mask** — HSV green threshold on the median background; largest component + hole filling.
2. **Background model** — per-pixel median of ~60 frames sampled across the clip.
3. **Foreground mask** — |blur(frame) − background| > τ inside the pitch mask, then morphology.
4. **Region proposals** — connected components filtered by a perspective size model h(y) fitted on training GT.

In [ ]:
"""Stage figure (requires Member 1's stage outputs; placeholder otherwise)."""
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from soccertrack.visualize import plot_stage_figure

stage_dir = OUTPUTS / 'stages' / '117093'
if stage_dir.exists():
    out_path = OUTPUTS / 'figures' / 'overlays' / '117093_stage.png'
    plot_stage_figure(stage_dir, frame_id=0, out_path=out_path)
    img = mpimg.imread(str(out_path))
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(img); ax.axis('off')
    plt.show()
else:
    print('[PLACEHOLDER] outputs/stages/117093/ not found — waiting for Member 1 M5.')

## 6. Patch Dataset and Class Imbalance

> **[PLACEHOLDER — Member 1 section]** Insert content from `docs/phase2/sections/member_1_scratch.md` §Patches.

- Patches: 64 × 32 px RGB crops.
- **1:3 positive:negative ratio** (GT crops : random windows + partial windows + unmatched proposals).
- Class imbalance handled via weighted cross-entropy (CNN) and `class_weight='balanced'` (SVM).

## 7. HOG + Linear SVM (Approach A — Classifier)

> **[PLACEHOLDER — Member 1 section]** Insert from `docs/phase2/sections/member_1_scratch.md` §HOG+SVM.  
> **[PLACEHOLDER — Member 3 E1 figures]** Confusion matrix, ROC curve, PR curve from `outputs/figures/eval/`.

In [ ]:
"""Load E1 metrics for HOG+SVM (fast mode: sample JSON)."""
metrics_dir = OUTPUTS / 'metrics'
e1_path = next(metrics_dir.glob('E1__scratch_hog_svm__val__*.json'), None) if metrics_dir.exists() else None

if e1_path:
    with e1_path.open() as fh:
        e1 = json.load(fh)
    m = e1['metrics']
    print(f"HOG+SVM E1 — Sensitivity: {m['sensitivity']:.4f}  Specificity: {m['specificity']:.4f}  "
          f"F1: {m['f1']:.4f}  ROC-AUC: {m['roc_auc']:.4f}")
else:
    print('[PLACEHOLDER] E1 HOG+SVM metrics not yet available — waiting for Member 1 M8 + Member 3 M1.')

## 8. Small CNN (Approach A — Classifier)

> **[PLACEHOLDER — Member 1 section]** Insert from `docs/phase2/sections/member_1_scratch.md` §CNN.  
> **[PLACEHOLDER — Member 3 E1 figures]** Learning curves, confusion matrix from `outputs/figures/eval/`.

Architecture: 3 × (Conv-BN-ReLU-MaxPool) → global average pooling → FC; ~25k parameters.  
Training: weighted cross-entropy, early stopping on validation loss.

In [ ]:
"""Load E1 metrics for Small CNN."""
e1_cnn_path = next(metrics_dir.glob('E1__scratch_cnn__val__*.json'), None) if metrics_dir.exists() else None

if e1_cnn_path:
    with e1_cnn_path.open() as fh:
        e1_cnn = json.load(fh)
    m = e1_cnn['metrics']
    print(f"Small CNN E1 — Sensitivity: {m['sensitivity']:.4f}  Specificity: {m['specificity']:.4f}  "
          f"F1: {m['f1']:.4f}  ROC-AUC: {m['roc_auc']:.4f}")
else:
    print('[PLACEHOLDER] E1 Small CNN metrics not yet available — waiting for Member 1 M9 + Member 3 M1.')

## 9. Pretrained Detector: Tiled YOLOv8n (Approach B)

> **[PLACEHOLDER — Member 2 section]** Insert from `docs/phase2/sections/member_2_yolo.md`.  

Key settings (from `configs/yolo.yaml`):
- Native-resolution overlapping tiles (default ~1280 px wide, ≥ 64 px overlap).
- COCO-pretrained YOLOv8n, class `person` only.
- Cross-tile NMS; optional pitch-mask filter.

In [ ]:
"""Load E3 metrics for YOLOv8n (fast mode: sample)."""
e3_yolo_path = next(metrics_dir.glob('E3__yolov8n_coco__val__*.json'), None) if metrics_dir.exists() else None

if e3_yolo_path:
    with e3_yolo_path.open() as fh:
        e3 = json.load(fh)
    m = e3['metrics']
    print(f"YOLOv8n E3 — P: {m['precision']:.4f}  R: {m['recall']:.4f}  "
          f"F1: {m['f1']:.4f}  AP@50: {m['ap50']:.4f}  FP/frame: {m['fp_per_frame']:.2f}")
else:
    print('[PLACEHOLDER] E3 YOLOv8n metrics not yet available — waiting for Member 2 + Member 3.')

## 10. Detection Results: Approach A vs Approach B (Validation)

In [ ]:
"""Load detection comparison table & evaluation figures (Member 3 compare.py)."""
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

table_path = OUTPUTS / 'tables' / 'detection_val.csv'
if table_path.exists():
    df = pd.read_csv(table_path)
    display(df)
else:
    print('[PLACEHOLDER] outputs/tables/detection_val.csv not yet available — run scripts/run_evaluation.py compare --split val.')

eval_fig_dir = OUTPUTS / 'figures' / 'eval'
for fig_name in ('pr_detection_val.png', 'recall_by_height_val.png'):
    fpath = eval_fig_dir / fig_name
    if fpath.exists():
        img = mpimg.imread(str(fpath))
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.imshow(img); ax.axis('off')
        plt.tight_layout(); plt.show()

In [ ]:
"""Show overlay sample (Approach A vs B on the same validation frame)."""
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

compare_dir = OUTPUTS / 'figures' / 'overlays'
compare_imgs = sorted(compare_dir.glob('*compare*.png')) if compare_dir.exists() else []

if compare_imgs:
    img = mpimg.imread(str(compare_imgs[0]))
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(img); ax.axis('off')
    ax.set_title('Pipeline A (top) vs Pipeline B (bottom) — same frame, validation match 117093')
    plt.tight_layout(); plt.show()
else:
    print('[PLACEHOLDER] No comparison overlay found. Run after Member 1+2 deliver track tables.')

## 11. Tracking: IoU+Hungarian (Pipeline A) and ByteTrack (Pipeline B)

> **[PLACEHOLDER — Members 1 & 2 sections]** Insert tracker descriptions.  
> **[PLACEHOLDER — Member 3 E4 tables]** HOTA, IDF1, MOTA, IDSW from `outputs/metrics/`.

E4 evaluation on a 60-second window (1500 frames) of the validation match, then on the two held-out test matches.  
Oracle rows (GT boxes as input) provide an upper bound on tracking performance.

In [ ]:
"""Load E4 tracking metrics & tracking comparison table."""
tracking_tbl = OUTPUTS / 'tables' / 'tracking_val.csv'
if tracking_tbl.exists():
    display(pd.read_csv(tracking_tbl))

e4_path = next(metrics_dir.glob('E4__*__val__*.json'), None) if metrics_dir.exists() else None

if e4_path:
    with e4_path.open() as fh:
        e4 = json.load(fh)
    m = e4['metrics']
    print(f"E4 ({e4['method']}) — HOTA: {m['hota']:.4f}  IDF1: {m['idf1']:.4f}  "
          f"MOTA: {m['mota']:.4f}  IDSW: {int(m['idsw'])}")
else:
    print('[PLACEHOLDER] E4 tracking metrics not yet available — waiting for Members 1/2 + Member 3.')

## 12. Frozen Test Results

> **[PLACEHOLDER — Member 3]** One logged test run per system on 128057 and 132831.  
> Numbers are read from `outputs/metrics/` — never entered manually.  
> The config SHA-256 and data hashes are recorded in `outputs/test_log.md`.

In [ ]:
"""Check test log existence."""
test_log = OUTPUTS / 'test_log.md'
if test_log.exists():
    print(test_log.read_text(encoding='utf-8'))
else:
    print('[PLACEHOLDER] outputs/test_log.md not found — frozen test run has not been executed yet.')

## 13. Comparison with the Paper and the SoccerTrack Challenge

> **[PLACEHOLDER — Member 3 comparison table]** Insert from `outputs/tables/references.csv`.

| Reference | Comparable? | Note |
|---|---|---|
| Paper dataset stats | ✅ reproducible | Median box difference noted |
| Challenge HOTA 0.59–0.60 | ✅ approximately | Same test clips; exact config undocumented |
| GTATrack ByteTrack 0.42 | ⚠️ indicative | Their validation split |
| Paper GS-HOTA / BAS mAP | ❌ | Different tasks (pitch-space / ball action) |

## 14. Limitations and Phase 3 Plan

### Limitations (Phase 2)
- Approach A struggles with tiny/merged players (median box ~20×45 px).
- ~90 % of false positives from YOLOv8n are **unlabelled people** (referees, staff) — explainable, not bugs.
- Tracking evaluated on a 60-second window only; full-clip evaluation is Phase 3.
- Calibration error (~2–3 m) is propagated to any pitch-space metric.
- Dataset: 4-minute clips only; no physiological ground truth; no injury/fatigue claims.

### Phase 3 Plan
1. **Failure analysis** on training/validation data → ranked improvement list.
2. **Improvements**: YOLO fine-tuning (I-1); Approach A hard negatives + blob splitting (I-2); Kalman tracker (I-3).
3. **Explainability**: Grad-CAM on CNN; HOG/SVM weight maps.
4. **Pitch mapping**: homography from keypoints → PitchTrack table.
5. **External workload**: Savitzky–Golay smoothing → speed, acceleration, heatmaps.
6. **Final frozen test run** on full clips; Δ table vs Phase 2.

> Phase 3 workload indicators are **external-load proxies only** (distance, speed, acceleration).  
> The dataset has no physiological fatigue or injury ground truth — none is claimed.